# EduGemma — Fine-Tune Gemma 4 E4B on STEM Education Data

This notebook fine-tunes Gemma 4 E4B on 500 curated STEM Q&A pairs using Unsloth.

**Setup:**
1. Settings → Accelerator → GPU T4 (or P100)
2. Settings → Internet → On (for pip install)
3. Add this notebook's data as a Kaggle dataset (upload `unsloth_training_data.jsonl`)

In [ ]:
# Install Unsloth and dependencies
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets

In [ ]:
from unsloth import FastModel
import torch
import json
import os

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if torch.cuda.is_available() else 'No GPU')

## Load Model

In [ ]:
# Load Gemma 4 E4B with 4-bit quantization (fits on T4 16GB)
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-e4b",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,  # Auto-detect
)
print('Model loaded successfully!')

In [ ]:
# Add LoRA adapters for efficient fine-tuning
model = FastModel.get_peft_model(
    model,
    r=16,           # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('LoRA adapters added!')

## Load Training Data

In [ ]:
from datasets import load_dataset

# Try loading from Kaggle dataset first, fallback to inline data
kaggle_data_path = "/kaggle/input/edugemma-training/unsloth_training_data.jsonl"

if os.path.exists(kaggle_data_path):
    print(f'Loading from Kaggle dataset: {kaggle_data_path}')
    dataset = load_dataset("json", data_files=kaggle_data_path, split="train")
else:
    # Look for the file in current directory
    local_path = "unsloth_training_data.jsonl"
    if os.path.exists(local_path):
        print(f'Loading from local file: {local_path}')
        dataset = load_dataset("json", data_files=local_path, split="train")
    else:
        raise FileNotFoundError(
            f"Training data not found. Upload unsloth_training_data.jsonl as a Kaggle dataset "
            f"named 'edugemma-training', or place it in the working directory."
        )

print(f'Training examples: {len(dataset)}')
print(f'Sample: {dataset[0]["text"][:200]}...')

## Fine-Tune

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/kaggle/working/outputs",
        report_to="none",  # Disable wandb/tensorboard
    ),
)

print(f'Starting training for 3 epochs on {len(dataset)} examples...')
print(f'Effective batch size: {2 * 4} = 8')
print(f'Approx steps: {(len(dataset) // 8) * 3} steps')

In [ ]:
# Train!
trainer.train()
print('Training complete!')

## Test the Fine-Tuned Model

In [ ]:
# Test with STEM questions
test_questions = [
    "How do I find the derivative of x^3?",
    "What is Newton's second law?",
    "Explain the difference between mitosis and meiosis.",
    "How do I solve a system of linear equations?",
    "What is the ideal gas law?",
]

for question in test_questions:
    messages = [
        {"role": "system", "content": "You are EduGemma, an adaptive STEM tutor. Explain concepts step by step with real-world examples."},
        {"role": "user", "content": question}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f'\nQ: {question}')
    print(f'A: {response[:300]}...')
    print('---')

## Save & Export

In [ ]:
# Save LoRA adapters
model.save_pretrained("/kaggle/working/edugemma-e4b-lora")
tokenizer.save_pretrained("/kaggle/working/edugemma-e4b-lora")
print('LoRA adapters saved!')

# Save merged model (for Ollama)
model.save_pretrained_merged("/kaggle/working/edugemma-e4b-merged", tokenizer)
print('Merged model saved!')

In [ ]:
# Export to GGUF for Ollama deployment
model.save_pretrained_gguf("/kaggle/working/edugemma-e4b", tokenizer, quantization_method="q4_k_m")
print('GGUF model exported! Download edugemma-e4b-q4_k_m.gguf for Ollama.')
print('\nTo use with Ollama:')
print('1. Create a Modelfile:')
print('   FROM edugemma-e4b-q4_k_m.gguf')
print('   TEMPLATE "{{- if .System }}<|im_start|>system')
print('   {{ .System }}<|im_end|>')
print('   {{- end }}')
print('   <|im_start|>user')
print('   {{ .Prompt }}<|im_end|>')
print('   <|im_start|>assistant')
print('   {{ .Response }}<|im_end|>"')
print('2. ollama create edugemma -f Modelfile')
print('3. ollama run edugemma')